<a href="https://colab.research.google.com/github/laisveloso/ceichina-colab/blob/main/Monitor_china_teste_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# importarção das bibliotecas e pacotes necessários
import pandas as pd

import matplotlib.pyplot as plt

import string
import re

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from google.colab import files
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.probability import FreqDist
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.metrics import confusion_matrix


!pip install leia-br
from LeIA import SentimentIntensityAnalyzer

nltk.download('all')
nltk.download('stopwords')


In [ ]:
from google.colab import files
import pandas as pd

# Upload do arquivo CSV
uploaded = files.upload()

# Lê o arquivo CSV (substitua 'nome_do_arquivo.csv' pelo nome do seu arquivo)
nome_do_arquivo = next(iter(uploaded))
df = pd.read_csv(nome_do_arquivo)

# Supondo que seu DataFrame já está carregado como 'df'
df = df.rename(columns={'tweet ': 'tweet'})
df['tweet'] = df['tweet'].astype(str)

# Mostra as primeiras linhas
df.head()


In [ ]:
def pre_processamento(texto):

    # seleciona apenas letras e coloca todas em minúsculo
    letras_min =  re.findall(r'\b[A-zÀ-úü]+\b', texto.lower())

    # remove stopwords
    stopwords = nltk.corpus.stopwords.words('portuguese')
    stop = set(stopwords)
    sem_stopwords = [w for w in letras_min if w not in stop]

    # juntando os tokens novamente em formato de texto
    texto_limpo = " ".join(sem_stopwords)

    return texto_limpo

In [ ]:
df['tokens'] = df['tweet'].apply(pre_processamento)
df.head()

In [ ]:
all_words = [word for tokens in df['tokens'] for word in tokens.split(" ")]

freq_dist = FreqDist(all_words)

print("🔠 Palavras mais frequentes:")
print(freq_dist.most_common(10))

# Plotagem do gráfico
plt.figure(figsize=(12, 6))
freq_dist.plot(20, title='Top 20 Palavras Mais Frequentes', cumulative=False)
plt.show()

In [ ]:
sia = SentimentIntensityAnalyzer()

def get_compound_Sentimento(texto):
  sentimento = sia.polarity_scores(texto)
  return sentimento['compound']

def get_neg_Sentimento(texto):
  sentimento = sia.polarity_scores(texto)
  return sentimento['neg']

def get_neu_Sentimento(texto):
  sentimento = sia.polarity_scores(texto)
  return sentimento['neu']

def get_pos_Sentimento(texto):
  sentimento = sia.polarity_scores(texto)
  return sentimento['pos']

def get_all_Sentimento(texto):
  sentimento = sia.polarity_scores(texto)
  return sentimento

In [ ]:
df['sentimento'] = df['tokens'].apply(get_compound_Sentimento)
df['negativo']  = df['tokens'].apply(get_neg_Sentimento)
df['positivo']  = df['tokens'].apply(get_pos_Sentimento)
df['neutro']    = df['tokens'].apply(get_neu_Sentimento)
df.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.histplot(df['sentimento'], bins=20, kde=True, color='skyblue', edgecolor='black')
plt.title('Distribuição da Pontuação de Sentimento (compound)', fontsize=14)
plt.xlabel('Pontuação compound (VADER)', fontsize=12)
plt.ylabel('Frequência', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
from matplotlib import pyplot as plt
df['sentimento'].plot(kind='hist', bins=20, title='sentimento')
plt.gca().spines[['top', 'right',]].set_visible(False)

In [ ]:
df['sentimento'] = df['tokens'].apply(get_compound_Sentimento)
df['negativo']  = df['tokens'].apply(get_neg_Sentimento)
df['positivo']  = df['tokens'].apply(get_pos_Sentimento)
df['neutro']    = df['tokens'].apply(get_neu_Sentimento)
df.head()

RANDOM FORREST

In [ ]:
nltk.download('stopwords')


In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    # Remove URLs, menções e hashtags
    text = re.sub(r'http\S+|@\w+|#\w+', '', text)
    # Remove pontuações
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Converte para minúsculas
    text = text.lower()
    # Remove números
    text = re.sub(r'\d+', '', text)
    # Remove stopwords
    stop_words = set(stopwords.words('portuguese'))
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return ' '.join(words)

In [ ]:
df['cleaned_tweet'] = df['tweet'].apply(clean_text)

# Separar dados rotulados e não rotulados
df_labeled = df[df['viés político'].notna()].copy()
df_unlabeled = df[df['viés político'].isna()].copy()


In [ ]:
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df_labeled['cleaned_tweet'])
y = df_labeled['viés político']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight='balanced',
    random_state=42
)
rf_model.fit(X_train, y_train)

In [ ]:
y_pred = rf_model.predict(X_test)
print("Acurácia:", accuracy_score(y_test, y_pred))
print("\nRelatório de Classificação:")
print(classification_report(y_test, y_pred))

In [ ]:
if not df_unlabeled.empty:
    X_unlabeled = vectorizer.transform(df_unlabeled['cleaned_tweet'])
    rf_predictions = rf_model.predict(X_unlabeled)
    df_unlabeled['viés político (RF)'] = rf_predictions

In [ ]:
df_final_rf = pd.concat([df_labeled, df_unlabeled], ignore_index=True)

# Exportar resultados
output_filename_rf = 'tweets_classificados_rf.csv'
df_final_rf.to_csv(output_filename_rf, index=False, encoding='utf-8-sig')

print("\nResumo da classificação por Random Forest:")
print(df_final_rf['viés político (RF)'].value_counts())

In [ ]:
pd.set_option('display.max_colwidth', None)
print( df_final_rf[df_final_rf['viés político (RF)'] == 'contra-israel']['cleaned_tweet']  )

In [ ]:
feature_names = vectorizer.get_feature_names_out()
importances = rf_model.feature_importances_
top_features = pd.Series(importances, index=feature_names).sort_values(ascending=False)[:20]
print("\nTop 20 features mais importantes:")
print(top_features)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix


In [ ]:
y_ori = y
y_pred = rf_model.predict(X)
cm = confusion_matrix(y_ori, y_pred)
classes = y_test.unique()

In [ ]:
plt.figure(figsize=(10, 8))

# Plotar heatmap
sns.heatmap(cm,
            annot=True,
            fmt='d',
            cmap='Blues',
            xticklabels=classes,
            yticklabels=classes,
            cbar=False)

# Adicionar rótulos e título
plt.title('Matriz de Confusão - Random Forest', fontsize=16, pad=20)
plt.xlabel('Predito', fontsize=14)
plt.ylabel('Real', fontsize=14)
plt.xticks(fontsize=12, rotation=45)
plt.yticks(fontsize=12, rotation=0)

total = cm.sum()
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j+0.5, i+0.3, f"{cm[i, j]}\n({cm[i, j]/total:.1%})",
                 ha='center', va='center', color='black', fontsize=10)

# Mostrar a plotagem
plt.tight_layout()
plt.savefig('matriz_confusao_rf.png', dpi=300)
plt.show()

plt.figure(figsize=(10, 8))
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

sns.heatmap(cm_norm,
            annot=True,
            fmt='.2%',
            cmap='Greens',
            xticklabels=classes,
            yticklabels=classes)

plt.title('Matriz de Confusão Normalizada - Random Forest', fontsize=16, pad=20)
plt.xlabel('Predito', fontsize=14)
plt.ylabel('Real', fontsize=14)
plt.xticks(fontsize=12, rotation=45)
plt.yticks(fontsize=12, rotation=0)
plt.tight_layout()
plt.savefig('matriz_confusao_normalizada_rf.png', dpi=300)
plt.show()